# Surper_GCG Colab 运行面板

这个 notebook 直接调用当前的 `run_pipeline.py` 入口。

推荐顺序：
1. 先跑 `baseline_diagnosis`，打开真正的 `12B` 主线入口
2. 把 `eval_calibration` / `Exp11` 视为可选的旧 review-pack 支线
3. 在推进后续机制阶段前，先读导出的结果文件
4. 输出目录写入 `results/pipeline_runs/`


## 1. 挂载 Drive

用 Google Drive 保存仓库副本和 pipeline 输出。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 克隆仓库并安装

把仓库克隆到 `/content`，然后在仓库根目录执行安装。

这个单元是幂等的：如果运行时里已经有仓库，就会跳过 clone。

In [ ]:
%cd /content
import os
if not os.path.exists('/content/Surper_GCG'):
    !git clone https://github.com/awaa-col/Surper_GCG_ResearchWorkFlow.git Surper_GCG
else:
    print('仓库已存在于 /content/Surper_GCG，跳过 clone。')
%cd /content/Surper_GCG
!pip install -e .

## 3. 配置运行环境

设置 Hugging Face 凭据、运行名、模型和共享参数。

In [ ]:
import os

HF_TOKEN = ''
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

BASELINE_RUN_NAME = 'gemma3_12b_main'
GATE_RUN_NAME = f'{BASELINE_RUN_NAME}_gate'
ACTIVE_RUN_NAME = GATE_RUN_NAME
RESUME = '--resume'  # 新开跑时改成 ''
MODEL = 'google/gemma-3-12b-it'
N_TRAIN = 64
N_EVAL = 2
MAX_NEW_TOKENS = 96


## 4. 查看可用 preset 和 stage

先列出当前 preset，再确认哪些 stage 已解锁、哪些仍被阻塞。

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py --list-presets

In [ ]:
!python run_pipeline.py --list-stages

## 5. 对主线入口做 Dry-Run

从 `baseline_diagnosis` 开始。这是第一个真正的 `12B` 主线科技点。

先做 dry-run，确认 preset、输出目录和生成命令都正确。

In [ ]:
!python run_pipeline.py \
  --preset baseline_diagnosis \
  --run-name "$BASELINE_RUN_NAME" \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## 6. 运行主线入口

默认主线先跑 `baseline_diagnosis`。

不要把 `eval_calibration` 当成真正的第一关，它只是 review 对齐用的辅助科技。

In [ ]:
!python run_pipeline.py \
  --preset baseline_diagnosis \
  --run-name "$BASELINE_RUN_NAME" \
  $RESUME \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## 6C0. 对 Gate Discovery 做 Dry-Run

在正式跑 gate 之前，先用这个单元确认 preset 和输出路径。

这里仍然只会进入已解锁的 `gate_discovery`，不会碰后面未解锁的阶段。

In [ ]:
!python run_pipeline.py \
  --preset gate_discovery_bootstrap \
  --run-name "$GATE_RUN_NAME" \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## 6C. 下一个已解锁科技点：Gate Discovery Bootstrap

只有在 baseline diagnosis 的结果看起来正常之后，才运行这一段。

这会启动第一轮 `12B-first` gate 扫描，但不会提前解锁 cross-layer、detect 或 late-family。

In [ ]:
!python run_pipeline.py \
  --preset gate_discovery_bootstrap \
  --run-name "$GATE_RUN_NAME" \
  $RESUME \
  --model "$MODEL" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## 6B. 可选支线：旧 Review-Pack 校准

只有在你确实想用旧结果生成手工 review pack 时，才运行这一段。

这和从零开始跑 `12B` 主线不是一回事。

In [ ]:
# 可选：只有当你的 Colab 环境里已经有 Exp11 所需的旧结果文件时才运行
!python run_pipeline.py \
  --preset eval_calibration \
  --run-name "${BASELINE_RUN_NAME}_eval_calibration" \
  $RESUME \
  --model "$MODEL"

In [ ]:
print('当前解锁目标：先确认 12B 原始 refusal object 和 baseline seam。只有确认后，才进入 gate discovery。')

## 7. 检查输出并打包到 Drive

先检查当前 run 目录，再只打包这一次运行的结果，方便之后下载。

In [ ]:
!ls -lah /content/Surper_GCG/results/pipeline_runs
!ls -lah /content/Surper_GCG/results/pipeline_runs/$ACTIVE_RUN_NAME

In [ ]:
!zip -r /content/drive/MyDrive/${ACTIVE_RUN_NAME}_pipeline_results.zip /content/Surper_GCG/results/pipeline_runs/$ACTIVE_RUN_NAME